In [ ]:
#data ldoing functions 

def load_images_from_folder(folder_path, class_names):
    # Local imports for data loading
    import os
    import numpy as np
    from tensorflow.keras.preprocessing.image import load_img, img_to_array

    images = []
    labels = []

    for label, emotion in enumerate(class_names):
        emotion_folder = os.path.join(folder_path, emotion)

        for image_name in os.listdir(emotion_folder):
            image_path = os.path.join(emotion_folder, image_name)

            try:
                img = load_img(image_path, target_size=(48, 48), color_mode='grayscale')
                img_array = img_to_array(img) / 255.0

                images.append(img_array)
                labels.append(label)

            except Exception as e:
                print(f"Error loading {image_path}: {e}")

    return np.array(images), np.array(labels)


In [ ]:
# Dataset paths
train_path = 'Emotion_dataset/train'
test_path = 'Emotion_dataset/test'

class_names = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

# Load training and test data
print("Loading training data...")
X_train, y_train = load_images_from_folder(train_path, class_names)

print("Loading test data...")
X_test, y_test = load_images_from_folder(test_path, class_names)

print("Training set shape:", X_train.shape)
print("Training labels shape:", y_train.shape)
print("Test set shape:", X_test.shape)
print("Test labels shape:", y_test.shape)

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Conv2D(64, 3, activation='relu', input_shape=(48, 48, 1), padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    
    layers.Conv2D(256, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(256, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(len(class_names), activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(0.001), 
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)


In [ ]:
#training 

# Import utilities required specifically for training and data augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)
datagen.fit(X_train)

# Train model
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=64),
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[
        ModelCheckpoint('best_emotion_model.keras', monitor='val_accuracy', save_best_only=True, mode='max'),
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)
    ]
)


In [ ]:
# Evaluation imports
import numpy as np
from sklearn.metrics import classification_report

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")

y_pred_classes = np.argmax(model.predict(X_test), axis=1)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes, target_names=class_names))


# Visualization imports
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('confusion_matrix.png')
plt.show()

# Training history plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss')
ax2.legend()
ax2.grid(True)
plt.savefig('training_history.png')
plt.show()


In [ ]:
# camera 
print("Starting camera - Press 'q' to quit")


# Import the computer vision package and reload components needed for application mode
import cv2
import numpy as np
from tensorflow import keras

# Load model
model = keras.models.load_model("best_emotion_model.keras")

class_names_display = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

# Load face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

# Start webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("❌ Error: Could not open camera")
    exit()

frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.2,
        minNeighbors=6,
        minSize=(30, 30)
    )

    # Run prediction every 3rd frame (for performance)
    if frame_count % 3 == 0:
        for (x, y, w, h) in faces:
            
            # Extract face ROI
            face_roi = gray[y:y+h, x:x+w]
            face_roi = cv2.resize(face_roi, (48, 48))
            face_roi = face_roi.astype("float32") / 255.0
            face_roi = face_roi.reshape(1, 48, 48, 1)

            # Predict
            pred = model.predict(face_roi, verbose=0)
            idx = np.argmax(pred)
            confidence = pred[0][idx]

            # Label
            label = f"{class_names_display[idx]} ({confidence*100:.1f}%)"

            # Draw rectangle
            cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

            # Draw label background
            cv2.rectangle(frame, (x, y-30), (x+w, y), (0, 255, 0), -1)

            # Put text
            cv2.putText(frame, label, (x+5, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                        (0, 0, 0), 2)

    # Show frame
    cv2.imshow("Emotion Detector", frame)

    # Exit on 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
cap.release()
cv2.destroyAllWindows()

print("\nEmotion detection ended!")